In [ ]:
#Import necessary libraries
import cv2 # OpenCV for image processing
import numpy as np # NumPy for numerical operations
import tensorflow as tf # TensorFlow for deep learning


In [3]:
# Load the saved 15-epoch tuned model from your POE folder
tuned_model = tf.keras.models.load_model(
    r"C:\Users\caden\OneDrive\Documents\Programming - PDAN8412\POE\tuned_face_cnn_15ep.keras"
)

In [ ]:
#Set the TensorFlow model to inference mode 

# Load OpenCV's built-in Haar Cascade face detector
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

# Connect to the default laptop webcam 
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Could not access the webcam.")
else:
    print("Webcam active. Press 'q' in the video window to quit.")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Mirror the camera view horizontally so it feels like a natural mirror
    frame = cv2.flip(frame, 1)
    frame_h, frame_w, _ = frame.shape

    # Convert to grayscale for fast face detection
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=5, minSize=(80, 80)
    )

    for (x, y, w, h) in faces:
        # Add 20% padding around the detected face so hair and chin framing
        # matches the FFHQ / StyleGAN training images
        pad_w = int(w * 0.20)
        pad_h = int(h * 0.20)
        x1 = max(0, x - pad_w)
        y1 = max(0, y - pad_h)
        x2 = min(frame_w, x + w + pad_w)
        y2 = min(frame_h, y + h + pad_h)

        # Crop the face region
        face_crop = frame[y1:y2, x1:x2]

        # Convert OpenCV's BGR color order to RGB (matching TensorFlow training data)
        face_rgb = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)

        # Resize to 128x128 and add batch dimension -> shape (1, 128, 128, 3)
        face_resized = cv2.resize(face_rgb, (128, 128))
        face_input = np.expand_dims(face_resized, axis=0)

        # Predict probability (0 = fake, 1 = real)
        # Note: Do NOT divide by 255 here because layers.Rescaling(1./255) is inside tuned_model
        prob = tuned_model.predict(face_input, verbose=0)[0][0]

        if prob > 0.5:
            label = f"REAL ({prob * 100:.1f}%)"
            color = (0, 255, 0)    # Green box in BGR
        else:
            label = f"FAKE ({(1 - prob) * 100:.1f}%)"
            color = (0, 0, 255)    # Red box in BGR

        # Draw bounding box and prediction label on the live video frame
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(
            frame, label, (x1, max(25, y1 - 10)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.75, color, 2
        )

    # Display the live feed
    cv2.imshow("Premises Security Prototype - Real vs. Fake Face Detection", frame)

    # Stop loop when 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

# Release camera hardware and close window
cap.release()
cv2.destroyAllWindows()

#References: 
#https://openaccess.thecvf.com/content_CVPR_2019/papers/Karras_A_Style-Based_Generator_Architecture_for_Generative_Adversarial_Networks_CVPR_2019_paper.pdf
#https://www.cs.cmu.edu/~efros/courses/LBMV07/Papers/viola-cvpr-01.pdf 
#https://mipro-proceedings.com/sites/mipro-proceedings.com/files/upload/sp/sp_008.pdf 

Webcam active. Press 'q' in the video window to quit.
